<a href="https://colab.research.google.com/github/peteraggi/ColabFavourites/blob/main/CoffeeCare_Dataset_Builder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CoffeeCare Dataset Builder
Downloads coffee leaf datasets, maps their classes to the CoffeeCare labels,
and builds fixed train / validation / test splits.

**Datasets**
- Soroti University, Uganda (Chelangat et al., 2025): Ugandan test set
- JMuBEN / JMuBEN2 (Kenya, Arabica)
- BRACOL (Brazil, Arabica)
- RoCoLe (Ecuador, Robusta)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

# Permanent storage on Google Drive
BASE_DIR = '/content/drive/MyDrive/ColabFavourites/CoffeeCare'
RAW_DIR = os.path.join(BASE_DIR, 'raw')
DATASET_DIR = os.path.join(BASE_DIR, 'dataset')
MODELS_DIR = os.path.join(BASE_DIR, 'models')

# Fast temporary storage on Colab (wiped when the session ends)
EXTRACTED_DIR = '/content/extracted'

for folder in [RAW_DIR, DATASET_DIR, MODELS_DIR, EXTRACTED_DIR]:
    os.makedirs(folder, exist_ok=True)
    print(f"Ready: {folder}")

Ready: /content/drive/MyDrive/ColabFavourites/CoffeeCare/raw
Ready: /content/drive/MyDrive/ColabFavourites/CoffeeCare/dataset
Ready: /content/drive/MyDrive/ColabFavourites/CoffeeCare/models
Ready: /content/extracted


In [3]:
SOROTI_URL = 'https://data.mendeley.com/public-api/zip/k36wnd6knb/download/1'
SOROTI_ZIP = os.path.join(RAW_DIR, 'soroti_uganda.zip')

if not os.path.exists(SOROTI_ZIP):
    !wget -q --show-progress -O "{SOROTI_ZIP}" "{SOROTI_URL}"
else:
    print('Already downloaded:', SOROTI_ZIP)

print('Size (MB):', round(os.path.getsize(SOROTI_ZIP) / 1024 / 1024, 1))

/content/drive/MyDr 100%[===================>]  24.82M  15.7MB/s    in 1.6s    
Size (MB): 24.8


In [4]:
import zipfile

SOROTI_DIR = os.path.join(EXTRACTED_DIR, 'soroti_uganda')

if not os.path.exists(SOROTI_DIR):
    with zipfile.ZipFile(SOROTI_ZIP, 'r') as z:
        z.extractall(SOROTI_DIR)
    print('Extracted to', SOROTI_DIR)
else:
    print('Already extracted:', SOROTI_DIR)

Extracted to /content/extracted/soroti_uganda


In [5]:
IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png')

def show_tree(root, max_depth=4):
    """Print the folder structure with the number of images in each folder"""
    for current, folders, files in os.walk(root):
        depth = current[len(root):].count(os.sep)
        if depth > max_depth:
            continue

        images = [f for f in files if f.lower().endswith(IMAGE_EXTENSIONS)]
        others = [f for f in files if not f.lower().endswith(IMAGE_EXTENSIONS)]

        indent = '    ' * depth
        print(f"{indent}{os.path.basename(current)}/  -> {len(images)} images, {len(others)} other files")

        if others and len(others) <= 5:
            print(f"{indent}    other files: {others}")

show_tree(SOROTI_DIR)

soroti_uganda/  -> 0 images, 0 other files
    A Machine  Learning Dataset for Classification of Common Coffee Leaf Diseases in Uganda/  -> 0 images, 0 other files
        coffee dataset/  -> 0 images, 0 other files
            phoma/  -> 1110 images, 0 other files
            leaf rust/  -> 1033 images, 0 other files
            Health leaves/  -> 1179 images, 0 other files


In [6]:
# 7 CoffeeCare classes (same keys as diseases.json in the backend)
COFFEECARE_CLASSES = [
    'brown_eye_spot',
    'coffee_leaf_rust',
    'healthy',
    'leaf_miner',
    'phoma_leaf_blight',
    'red_spider_mite',
    'unknown'
]

# Soroti folder names (lowercase) -> CoffeeCare classes
SOROTI_CLASS_MAP = {
    'health leaves': 'healthy',
    'leaf rust': 'coffee_leaf_rust',
    'phoma': 'phoma_leaf_blight'
}

# Safety check: every mapped label must be one of our classes
for label in SOROTI_CLASS_MAP.values():
    assert label in COFFEECARE_CLASSES, f"Unknown label: {label}"

print('Class map OK')

Class map OK
